In [10]:
# Installations
%pip install -q \
    pypdf \
    langchain-text-splitters \
    sentence-transformers \
    faiss-cpu \
    ollama \
    fastapi \
    uvicorn \
    requests \
    httpx \
    pydantic \
    nest-asyncio \
    streamlit \
    pandas \
    numpy \
    scikit-learn \
    matplotlib \
    python-dotenv \
    pytest

In [11]:
# Libraries
# Standard Library
import os
import sys
import json
import time
import asyncio
import logging
from pathlib import Path
from typing import Optional, List, Dict, Any
from contextlib import asynccontextmanager

# Data Processing
import numpy as np
import pandas as pd

# PDF Processing
from pypdf import PdfReader

# Text Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector Database
import faiss

# LLM - Ollama
import ollama

# FastAPI
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn

# HTTP Requests
import requests
import httpx

# Async support for notebooks
import nest_asyncio

# Evaluation
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

# Visualization
import matplotlib.pyplot as plt

# Environment Variables
from dotenv import load_dotenv

# Testing
import pytest

print("All imports completed successfully!")

All imports completed successfully!


In [38]:
 # Dataset

PDF_PATH = "/content/Diabetes_RAG_Knowledge_Base.pdf"
reader = PdfReader(PDF_PATH)
pages_text = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        pages_text.append({
            "page": page_number,
            "text": text
        })
print("Total PDF pages:", len(reader.pages))
print("Pages with extracted text:", len(pages_text))

# Ensure all text is read
print(pages_text[0]["text"][:2000])

Total PDF pages: 8
Pages with extracted text: 8
Diabetes RAG Knowledge Base  Educational use only
Page 1
 DIABETES RAG
KNOWLEDGE BASE
Structured educational reference for retrieval-augmented generation
Educational use only. Not a substitute for professional medical advice, diagnosis, or treatment. No individualized
diagnosis, prescriptions, insulin dosing, or medication adjustments. For urgent symptoms, seek immediate medical help.



In [63]:

cleaned_pages = []

for page in pages_text:

    text = page["text"]

    # Remove the evaluation question bank and everything after it
    text = re.split(
        r"Question bank for retrieval evaluation",
        text,
        flags=re.IGNORECASE
    )[0]

    # Remove page footer
    text = re.sub(
        r"Diabetes RAG Knowledge Base.*?Page\s*\d+",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    if text:
        cleaned_pages.append({
            "page": page["page"],
            "text": text
        })

print("Pages after cleaning:", len(cleaned_pages))

Pages after cleaning: 8


In [99]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

import re

chunks = []

for page in cleaned_pages:
    text = page["text"]
    page_num = page["page"]

    # Split content at section IDs such as DM-001, DM-002...
    sections = re.split(
        r"(?=DM-\d{3}:)",
        text
    )

    for section in sections:
        section = section.strip()

        if not section:
            continue

        # Identify section ID
        match = re.match(r"(DM-\d{3})", section)
        section_id = match.group(1) if match else "GENERAL"

        # Split each section into smaller chunks
        section_chunks = text_splitter.split_text(section)

        for chunk_text in section_chunks:
            chunks.append({
                "text": chunk_text,
                "page": page_num,
                "section_id": section_id
            })

print("Total chunks:", len(chunks))
print("\nSample:")
print(chunks[0])

Total chunks: 25

Sample:
{'text': 'Diabetes RAG Knowledge Base \x7f Educational use only Page 1 DIABETES RAG KNOWLEDGE BASE Structured educational reference for retrieval-augmented generation Educational use only. Not a substitute for professional medical advice, diagnosis, or treatment. No individualized diagnosis, prescriptions, insulin dosing, or medication adjustments. For urgent symptoms, seek immediate medical help.', 'page': 1, 'section_id': 'GENERAL'}


In [100]:
# Chunk Sample
print("First chunk:")
print(chunks[0]["text"])

print("\nSource page:", chunks[0]["page"])

First chunk:
Diabetes RAG Knowledge Base  Educational use only Page 1 DIABETES RAG KNOWLEDGE BASE Structured educational reference for retrieval-augmented generation Educational use only. Not a substitute for professional medical advice, diagnosis, or treatment. No individualized diagnosis, prescriptions, insulin dosing, or medication adjustments. For urgent symptoms, seek immediate medical help.

Source page: 1


In [101]:
# Embedding Model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [102]:
# Chnunks to embeddings

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (25, 384)


In [103]:
# Vector Dataset
# Get embedding dimension
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created.")
print("Vectors stored:", index.ntotal)

FAISS index created.
Vectors stored: 25


In [104]:
# Save Faiss Index and Chunks

save_dir = Path("/content/diabetes_rag_index")
save_dir.mkdir(parents=True, exist_ok=True)

faiss.write_index(
    index,
    str(save_dir / "index.faiss")
)

with open(save_dir / "chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("FAISS index and chunks saved.")


FAISS index and chunks saved.


In [105]:
# Testing search int he dataset
query = "What is the difference between type 1 and type 2 diabetes?"

# Convert query into an embedding
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Search FAISS
k = min(3, index.ntotal)

scores, indices = index.search(query_embedding, k)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):
    if idx == -1:
        continue

    print(f"\nResult {rank}")
    print(f"Similarity: {score:.4f}")
    print(f"Page: {chunks[idx]['page']}")
    print(chunks[idx]["text"])
    print("-" * 60)


Result 1
Similarity: 0.7182
Page: 2
DM-002: Types of diabetes Topic: Classification Source: ADA (2024), CDC (2024) Type 1 diabetes (T1D) An autoimmune condition in which the immune system attacks and destroys insulin-producing pancreatic beta cells, causing absolute insulin deficiency. It often begins in childhood or adolescence but can occur at any age. Type 2 diabetes (T2D) Progressive inadequate insulin secretion, commonly associated with insulin resistance. The supplied reference states it accounts for approximately 90–95% of diabetes cases. Gestational diabetes (GDM) Diabetes diagnosed during pregnancy; the reference treats it as a distinct category. How do T1D and T2D differ? T1D involves autoimmune beta-cell destruction and absolute insulin deficiency
------------------------------------------------------------

Result 2
Similarity: 0.5653
Page: 2
. How do T1D and T2D differ? T1D involves autoimmune beta-cell destruction and absolute insulin deficiency. T2D commonly begins with

In [106]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 108 not upgraded.


In [107]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [108]:
!ollama serve > /tmp/ollama.log 2>&1 &

In [109]:
!ollama --version

ollama version is 0.34.2


In [110]:
import time
time.sleep(5)

!ollama --version
!ollama list

ollama version is 0.34.2
NAME           ID              SIZE      MODIFIED      
llama3.2:1b    baf6a787fdff    1.3 GB    9 minutes ago    


In [111]:
!ollama pull llama3.2:1b

In [112]:
!ollama list

NAME           ID              SIZE      MODIFIED               
llama3.2:1b    baf6a787fdff    1.3 GB    Less than a second ago    


In [113]:
# testing Ollama
response = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "user",
            "content": "Explain diabetes in one simple sentence."
        }
    ]
)

print(response["message"]["content"])

Diabetes is a chronic condition where the body's ability to regulate blood sugar levels is impaired, leading to high blood sugar levels that can cause damage to organs and tissues over time.


In [143]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve more candidates than needed
    scores, indices = index.search(
        query_embedding,
        min(k * 3, len(chunks))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        chunk = chunks[idx]

        results.append({
            "text": chunk["text"],
            "page": chunk["page"],
            "section_id": chunk.get("section_id", "GENERAL"),
            "score": float(score)
        })

    # Keep the most relevant k results
    return results[:k]

In [132]:
# test
question = "How do type 1 and type 2 diabetes differ?"

results = retrieve(question, k=3)

for i, result in enumerate(results, start=1):

    print(f"\nResult {i}")
    print("Similarity:", round(result["score"], 4))
    print("Page:", result["page"])
    print(result["text"])
    print("-" * 60)


Result 1
Similarity: 0.7085
Page: 2
DM-002: Types of diabetes Topic: Classification Source: ADA (2024), CDC (2024) Type 1 diabetes (T1D) An autoimmune condition in which the immune system attacks and destroys insulin-producing pancreatic beta cells, causing absolute insulin deficiency. It often begins in childhood or adolescence but can occur at any age. Type 2 diabetes (T2D) Progressive inadequate insulin secretion, commonly associated with insulin resistance. The supplied reference states it accounts for approximately 90–95% of diabetes cases. Gestational diabetes (GDM) Diabetes diagnosed during pregnancy; the reference treats it as a distinct category. How do T1D and T2D differ? T1D involves autoimmune beta-cell destruction and absolute insulin deficiency
------------------------------------------------------------

Result 2
Similarity: 0.5796
Page: 2
. How do T1D and T2D differ? T1D involves autoimmune beta-cell destruction and absolute insulin deficiency. T2D commonly begins with

In [151]:
import re

def answer_question(question, k=3):

    results = retrieve(question, k=k)

    if not results:
        return {
            "answer": (
                "I don't have enough information in the "
                "retrieved sources to answer this question."
            ),
            "sources": []
        }

    # Normalize text for matching
    def normalize(text):
        return re.sub(r"[^a-z0-9\s]", " ", text.lower())

    question_words = set(normalize(question).split())

    # Remove common words that don't help identify the topic
    stop_words = {
        "what", "is", "are", "the", "a", "an", "of",
        "for", "to", "in", "on", "and", "how", "does",
        "do", "can", "with", "this", "that", "it"
    }

    question_words -= stop_words

    # Rank retrieved chunks by question-word overlap
    ranked_results = []

    for result in results:
        text = normalize(result["text"])
        text_words = set(text.split())

        overlap = len(question_words & text_words)

        ranked_results.append((
            overlap,
            result
        ))

    ranked_results.sort(
        key=lambda item: (
            item[0],
            item[1]["score"]
        ),
        reverse=True
    )

    selected_results = [
        result
        for _, result in ranked_results[:k]
    ]

    context = "\n\n".join(
        f"[Source {i} | Page {r['page']}]\n{r['text']}"
        for i, r in enumerate(selected_results, start=1)
    )

    system_prompt = """
You are a diabetes educational assistant.

Answer the exact question using ONLY the retrieved context.

Instructions:
- Identify the specific information requested.
- Use the context that directly answers that request.
- Do not replace requested information with a general definition.
- Include all relevant details available in the context.
- Do not add unsupported medical facts.
- If the context does not contain the answer, say so.
- Do not invent source labels or page numbers.
- This is educational information, not medical advice.

Return only the answer.
Do not generate a SOURCES section.
"""

    response = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"Retrieved context:\n{context}"
                )
            }
        ],
        options={"temperature": 0}
    )

    answer = response["message"]["content"]

    sources = [
        {
            "source": f"Source {i}",
            "page": r["page"],
            "similarity": round(r["score"], 4),
            "section_id": r.get("section_id", "GENERAL"),
            "text": r["text"]
        }
        for i, r in enumerate(selected_results, start=1)
    ]

    return {
        "answer": answer,
        "sources": sources
    }

In [152]:


question = "How do type 1 and type 2 diabetes differ?"

result = answer_question(question, k=3)

print("ANSWER:\n")
print(result["answer"])

print("\nRETRIEVED SOURCES:")

for source in result["sources"]:

    print(
        f"\n{source['source']} | "
        f"Page {source['page']} | "
        f"Similarity: {source['similarity']}"
    )

    print(source["text"])
    print("-" * 60)

ANSWER:

Type 1 diabetes (T1D) and type 2 diabetes (T2D) differ in the following ways:

1. Cause: Type 1 diabetes is an autoimmune condition in which the immune system attacks and destroys insulin-producing pancreatic beta cells, causing absolute insulin deficiency. Type 2 diabetes is a progressive inadequate insulin secretion, commonly associated with insulin resistance.
2. Age of onset: Type 1 diabetes often begins in childhood or adolescence, while type 2 diabetes can occur at any age.
3. Insulin deficiency: In type 1 diabetes, there is absolute insulin deficiency, while in type 2 diabetes, there is often insufficient insulin secretion.

RETRIEVED SOURCES:

Source 1 | Page 2 | Similarity: 0.7085
DM-002: Types of diabetes Topic: Classification Source: ADA (2024), CDC (2024) Type 1 diabetes (T1D) An autoimmune condition in which the immune system attacks and destroys insulin-producing pancreatic beta cells, causing absolute insulin deficiency. It often begins in childhood or adolescen

In [153]:
result = answer_question(question, k=1)

print("ANSWER:\n")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(
        f"{source['source']} | "
        f"Page {source['page']} | "
        f"Similarity: {source['similarity']}"
    )

ANSWER:

Type 1 diabetes (T1D) and type 2 diabetes (T2D) differ in the underlying cause and the body's response to insulin.

SOURCES:
Source 1 | Page 2 | Similarity: 0.7085


In [154]:
test_questions = [
    "What is diabetes mellitus?",
    "How do type 1 and type 2 diabetes differ?",
    "What are the risk factors for type 2 diabetes?",
    "What is insulin resistance?",
    "What are the symptoms of diabetes?"
]

for question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)

    result = answer_question(question, k=1)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCE PAGES:")
    for source in result["sources"]:
        print(
            f"{source['source']} | "
            f"Page {source['page']} | "
            f"Similarity: {source['similarity']}"
        )


QUESTION: What is diabetes mellitus?

ANSWER:
Diabetes mellitus is a chronic metabolic disease characterized by elevated blood glucose.

SOURCE PAGES:
Source 1 | Page 2 | Similarity: 0.7284

QUESTION: How do type 1 and type 2 diabetes differ?

ANSWER:
Type 1 diabetes (T1D) and type 2 diabetes (T2D) differ in the underlying cause and the body's response to insulin.

SOURCE PAGES:
Source 1 | Page 2 | Similarity: 0.7085

QUESTION: What are the risk factors for type 2 diabetes?

ANSWER:
The risk factors for type 2 diabetes include:

- Insulin resistance
- Obesity
- Physical inactivity
- Family history of type 2 diabetes
- Age (risk increases with age)
- Ethnicity (some ethnic groups are at higher risk)
- Certain medical conditions, such as polycystic ovary syndrome (PCOS) and Cushing's syndrome

SOURCE PAGES:
Source 1 | Page 2 | Similarity: 0.5979

QUESTION: What is insulin resistance?

ANSWER:
Insulin resistance is when cells in muscle, fat, and liver do not respond well to insulin.

SOU

In [155]:

question = "What is diabetes mellitus?"

results = retrieve(question, k=3)

print("Number of results:", len(results))

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Similarity:", round(result["score"], 4))
    print("Page:", result["page"])
    print(result["text"])
    print("-" * 50)

Number of results: 3

Result 1
Similarity: 0.7284
Page: 2
DM-001: Definition and core concepts Topic: Overview Source: WHO (2023), ADA (2024) What is diabetes mellitus? A chronic metabolic disease characterized by elevated blood glucose. It occurs when the pancreas does not produce enough insulin or when the body cannot effectively use the insulin it produces. What does insulin do? Insulin is a hormone that helps regulate blood glucose by facilitating glucose uptake into cells for energy. Why can chronic hyperglycemia be harmful? Over time, uncontrolled high blood glucose can damage the heart, blood vessels, eyes, kidneys, and nerves.
--------------------------------------------------

Result 2
Similarity: 0.5697
Page: 2
DM-002: Types of diabetes Topic: Classification Source: ADA (2024), CDC (2024) Type 1 diabetes (T1D) An autoimmune condition in which the immune system attacks and destroys insulin-producing pancreatic beta cells, causing absolute insulin deficiency. It often begins in

In [157]:
question = "What are the risk factors for type 2 diabetes?"

result = answer_question(question, k=3)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(
        source["section_id"],
        "| Page:", source["page"],
        "| Similarity:", source["similarity"]
    )

ANSWER:
The risk factors for type 2 diabetes are:

1. Overweight or obesity
2. Age 45 years or older
3. Family history of type 2 diabetes

SOURCES:
DM-002 | Page: 2 | Similarity: 0.5979
DM-003 | Page: 2 | Similarity: 0.5831
DM-005 | Page: 3 | Similarity: 0.4962


In [158]:
test_questions = [
    "What is diabetes mellitus?",
    "How do type 1 and type 2 diabetes differ?",
    "What are the symptoms of diabetes?",
    "What is insulin resistance?",
    "What is the A1C threshold for diabetes?",
    "What is hypoglycemia?"
]

for question in test_questions:
    result = answer_question(question, k=3)

    print("\nQUESTION:", question)
    print("ANSWER:", result["answer"])
    print("SOURCES:", [
        s["section_id"] for s in result["sources"]
    ])
    print("-" * 60)


QUESTION: What is diabetes mellitus?
ANSWER: Diabetes mellitus is a chronic metabolic disease characterized by elevated blood glucose. It occurs when the pancreas does not produce enough insulin or when the body cannot effectively use the insulin it produces.
SOURCES: ['DM-001', 'DM-002', 'DM-005']
------------------------------------------------------------

QUESTION: How do type 1 and type 2 diabetes differ?
ANSWER: Type 1 diabetes (T1D) and type 2 diabetes (T2D) differ in the following ways:

1. Cause: Type 1 diabetes is an autoimmune condition in which the immune system attacks and destroys insulin-producing pancreatic beta cells, causing absolute insulin deficiency. Type 2 diabetes is a progressive inadequate insulin secretion, commonly associated with insulin resistance.
2. Age of onset: Type 1 diabetes often begins in childhood or adolescence, while type 2 diabetes can occur at any age.
3. Insulin deficiency: In type 1 diabetes, there is absolute insulin deficiency, while in ty

In [159]:
question = "What are the warning signs of hypoglycemia?"

result = answer_question(question, k=3)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(
        source["section_id"],
        "| Page:", source["page"],
        "| Similarity:", source["similarity"]
    )

ANSWER:
The warning signs of hypoglycemia are:

- Shakiness
- Sweating
- Hunger
- Palpitations
- Dizziness
- Confusion
- Weakness

SOURCES:
DM-012 | Page: 5 | Similarity: 0.6479
DM-013 | Page: 5 | Similarity: 0.4859
DM-004 | Page: 3 | Similarity: 0.4858
